In [ ]:
# FASE 6 — DEPLOYMENT | CardioRisk · IBM Data Science · CRISP-DM
# Reporte clínico final, función de predicción deployable y recomendaciones

!pip install kagglehub imbalanced-learn xgboost --quiet

import kagglehub, os, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib
matplotlib.rcParams['figure.facecolor'] = '#0a0f1a'
matplotlib.rcParams['axes.facecolor']   = '#0d1526'
matplotlib.rcParams['text.color']       = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor']  = '#e2e8f0'
matplotlib.rcParams['xtick.color']      = '#7a8fa8'
matplotlib.rcParams['ytick.color']      = '#7a8fa8'
matplotlib.rcParams['axes.edgecolor']   = '#1a2c3d'
matplotlib.rcParams['grid.color']       = '#1a2c3d'
warnings.filterwarnings('ignore')
print("✓ Entorno listo")

In [ ]:
# ── BLOQUE 1: PIPELINE COMPLETO FINAL ──
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import recall_score, roc_auc_score, f1_score, confusion_matrix

KAGGLE_DATASET = "jocelyndumlao/cardiovascular-disease-dataset"
try:
    path = kagglehub.dataset_download(KAGGLE_DATASET)
    csv_path = next(f for f in [os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs] if f.endswith('.csv'))
    df = pd.read_csv(csv_path)
except:
    df = pd.read_csv('/content/cardiovascular_disease_dataset.csv')

df.columns = df.columns.str.lower().str.strip()
df = df.dropna()

CATEGORICAL  = ['gender','chestpain','restingelectro']
TARGET       = 'target'
df_enc       = pd.get_dummies(df, columns=CATEGORICAL, drop_first=True)
feature_cols = [c for c in df_enc.columns if c != TARGET]
X = df_enc[feature_cols]
y = df_enc[TARGET]

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_sc, y_train)

try:
    best_model = joblib.load('/content/best_model.pkl')
    print("✓ Modelo cargado desde F4/F5")
except:
    best_model = RandomForestClassifier(n_estimators=400, random_state=42)
    best_model.fit(X_train_sm, y_train_sm)
    print("✓ Modelo reentrenado")

y_pred = best_model.predict(X_test_sc)
y_prob = best_model.predict_proba(X_test_sc)[:,1]
recall  = recall_score(y_test, y_pred)
auc     = roc_auc_score(y_test, y_prob)
f1      = f1_score(y_test, y_pred)
cm      = confusion_matrix(y_test, y_pred)
tn,fp,fn,tp = cm.ravel()
print(f"Recall={recall:.4f} | AUC={auc:.4f} | F1={f1:.4f} | FN={fn}")

In [ ]:
# ── BLOQUE 2: DASHBOARD DE RESULTADOS FINALES ──
from sklearn.metrics import roc_curve

fig = plt.figure(figsize=(14, 9))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=.4, wspace=.35)

ax0 = fig.add_subplot(gs[0, :2])
ax0.axis('off')
kpis = [
    ("Recall",    f"{recall:.3f}", "#00f2fe",  "Métrica primaria (min. FN)"),
    ("AUC-ROC",   f"{auc:.3f}",   "#8b5cf6",  "Discriminación global"),
    ("F1-Score",  f"{f1:.3f}",    "#00ff88",  "Balance Prec/Recall"),
    ("Falsos-",   str(fn),        "#ff3355",  "Pacientes críticos no detectados"),
]
for i,(label,val,color,desc) in enumerate(kpis):
    x = i * .25
    ax0.add_patch(plt.Rectangle((x,.05),.22,.9,fill=True,facecolor='#0d1526',edgecolor=color,lw=1.5,transform=ax0.transAxes,clip_on=False))
    ax0.text(x+.11, .72, val,   ha='center',va='center',fontsize=22,fontweight='bold',color=color,transform=ax0.transAxes)
    ax0.text(x+.11, .42, label, ha='center',va='center',fontsize=12,color='#e2e8f0',transform=ax0.transAxes)
    ax0.text(x+.11, .18, desc,  ha='center',va='center',fontsize=8, color='#7a8fa8',transform=ax0.transAxes)
ax0.set_title('CardioRisk — Resultados Finales (Test Set)', fontsize=14, fontweight='bold', pad=12)

ax1 = fig.add_subplot(gs[0, 2])
ax1.imshow(cm, cmap='Blues', alpha=.7)
for i in range(2):
    for j in range(2):
        c = '#ff3355' if (i==1 and j==0) else '#e2e8f0'
        ax1.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=20,fontweight='bold',color=c)
ax1.set_xticks([0,1]); ax1.set_yticks([0,1])
ax1.set_xticklabels(['Bajo','Alto']); ax1.set_yticklabels(['Bajo','Alto'])
ax1.set_xlabel('Pred'); ax1.set_ylabel('Real'); ax1.set_title('Conf. Matrix')

ax2 = fig.add_subplot(gs[1, 0])
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr,tpr,color='#00f2fe',lw=2,label=f'AUC={auc:.3f}')
ax2.plot([0,1],[0,1],'--',color='#3a5570',lw=1)
ax2.fill_between(fpr,tpr,alpha=.07,color='#00f2fe')
ax2.set_title('Curva ROC'); ax2.legend(fontsize=9); ax2.grid(alpha=.3)

ax3 = fig.add_subplot(gs[1, 1:])
if hasattr(best_model,'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_,index=feature_cols).sort_values()
    colors_bar = ['#ff3355' if 'slope' in n or 'chest' in n or 'oldpeak' in n else '#00f2fe' for n in imp[-10:].index]
    imp[-10:].plot(kind='barh',ax=ax3,color=colors_bar,alpha=.8)
    ax3.set_title('Top 10 Predictores (Gini)'); ax3.grid(axis='x',alpha=.3)

plt.suptitle('CardioRisk — Dashboard Clínico Final', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('/content/f6_dashboard_final.png', dpi=150, bbox_inches='tight', facecolor='#0a0f1a')
plt.show()
print("✓ Dashboard guardado: /content/f6_dashboard_final.png")

In [ ]:
# ── BLOQUE 3: FUNCIÓN DE PREDICCIÓN DEPLOYABLE ──

def predecir_riesgo_cardiovascular(paciente: dict, modelo=best_model, sc=scaler, cols=feature_cols) -> dict:
    """
    Predice riesgo cardiovascular isquémico para un nuevo paciente.

    Parámetros esperados (dict):
      age, restingBP, serumcholestrol, maxheartrate, oldpeak,
      noofmajorvessels, gender, chestpain, restingelectro,
      exerciseangia, slope, fastingbloodsugar

    Retorna dict con: riesgo (0/1), probabilidad, nivel, recomendacion
    """
    df_p = pd.DataFrame([paciente])
    df_p.columns = df_p.columns.str.lower().str.strip()

    for cat in ['gender','chestpain','restingelectro']:
        if cat in df_p.columns:
            df_p = pd.get_dummies(df_p, columns=[cat], drop_first=True)

    for col in cols:
        if col not in df_p.columns:
            df_p[col] = 0
    df_p = df_p[cols]

    X_p = sc.transform(df_p.values)
    riesgo = int(modelo.predict(X_p)[0])
    prob   = float(modelo.predict_proba(X_p)[0][1])

    if prob < 0.35:
        nivel = "BAJO"
        rec   = "Control rutinario anual. Continuar hábitos saludables."
    elif prob < 0.65:
        nivel = "MODERADO"
        rec   = "Seguimiento semestral. Perfil lipídico y ECG recomendados."
    else:
        nivel = "ALTO"
        rec   = "Derivación urgente a cardiología. Ergometría y biomarcadores."

    return {"riesgo": riesgo, "probabilidad": round(prob, 4),
            "nivel": nivel, "recomendacion": rec}

# Ejemplo clínico
paciente_ejemplo = {
    "age": 58, "restingBP": 145, "serumcholestrol": 270,
    "maxheartrate": 132, "oldpeak": 2.1, "noofmajorvessels": 2,
    "gender": 1, "chestpain": 3, "restingelectro": 1,
    "exerciseangia": 1, "slope": 2, "fastingbloodsugar": 1
}

resultado = predecir_riesgo_cardiovascular(paciente_ejemplo)
print("PREDICCIÓN CLÍNICA — PACIENTE EJEMPLO")
print("─" * 40)
print(f"Riesgo:        {'ALTO RIESGO' if resultado['riesgo']==1 else 'BAJO RIESGO'}")
print(f"Probabilidad:  {resultado['probabilidad']:.1%}")
print(f"Nivel:         {resultado['nivel']}")
print(f"Recomendación: {resultado['recomendacion']}")

In [ ]:
# ── BLOQUE 4: REPORTE CLÍNICO FINAL ──
print("""
╔══════════════════════════════════════════════════════════════════╗
║         CARDIORISK — REPORTE CLÍNICO FINAL                      ║
║         IBM Data Science Professional Certificate · CRISP-DM    ║
╚══════════════════════════════════════════════════════════════════╝
""")

print(f"""RESULTADOS FINALES (Test Set — datos no vistos)
─────────────────────────────────────────────────────────────────
  Recall (Sensibilidad):  {recall:.4f}  ← KPI primario {'✓ ALCANZADO' if recall>=0.75 else '⚠ revisar umbral'}
  AUC-ROC:                {auc:.4f}
  F1-Score:               {f1:.4f}
  Falsos Negativos:       {fn}  pacientes críticos no detectados

PREDICTORES MÁS RELEVANTES (Spearman + SHAP)
─────────────────────────────────────────────────────────────────
  1. slope (ST descendente)   → Isquemia obstructiva severa
  2. chestpain (asintomático) → Riesgo silente — paradoja clínica
  3. oldpeak (depresión ST)   → Insuficiencia coronaria funcional
  4. noofmajorvessels         → Daño estructural multivaso
  5. maxheartrate (inverso)   → Reserva cardíaca reducida

IMPLICACIONES CLÍNICAS
─────────────────────────────────────────────────────────────────
  • El modelo prioriza Recall porque un Falso Negativo en riesgo
    cardiovascular tiene consecuencias potencialmente fatales.
  • La pendiente ST descendente es el predictor más robusto.
  • El dolor asintomático confirma la paradoja clínica:
    ausencia de dolor ≠ ausencia de riesgo.

─────────────────────────────────────────────────────────────────
juanjararPYBM · IBM Data Science Professional Certificate · 2025
─────────────────────────────────────────────────────────────────""")
print("→ FASE 6 COMPLETA — PROYECTO CARDIORISK TERMINADO ✓")